# Kilgarriff (1997: 233): ‘any difference in the linguistic character of two corpora will leave its trace in differences between their word frequency lists ... the individual meanings of texts are taken out of focus, to be replaced by the character of the whole’ (ibid.: 232).

Parts of Speech for identifying student performance information: <br>
<br>
1. wh-pronoun, possessive : "**whose**" : indicates responsibility <br>
2. adverb, superlative : "she finished the exercise **fastest**" : indicates relative performance <br>
3. predeterminer : "**all**" as in "**all** their assignments" : quantifies nouns <br>
4. adjective, superlative : "**kindest**" : indicates relative performance <br>
5. adverb, comparative : "**better**" : indicates relative performance<br>
6. possessive ending : "student **'s** grades" : indicates responsibility <br>
7. verb, gerund or present participle : "currently **failing**" : indicates present performance <br>
8. verb, 3rd person singular present : "your daughter **listens**" : indicates general performance <br>
9. pronoun, possessive : "**their** homework" : indicates responsibility

Vocabulary for identifying student performance information <br>
<br>
1. positive sentiment words, 'AMAZING' <br>
2. performance verbs, 'achieve' <br>
3. performance nouns, 'scholarship', 'honors' <br>
4. subject area / discipline words <br>
5. assignment and assessment words <br>
6. school schedule words <br>
7. explicit grades like 'A+' or '74'

# Data Preprocessing

In [ ]:
#Import packages
import pandas as pd
import numpy as np
import spacy
import re

In [ ]:
#Load spaCy
nlp = spacy.load('en_core_web_sm')

In [ ]:
#Load sample of 100k messages tagged 'academic'
acaDF = pd.read_csv('data/tpm_academic_100k.csv')
acaDF.sample(5)

In [ ]:
#Load sample of 100k random messages
ranDF = pd.read_csv('data/tpm_random_100k.csv')
ranDF.sample(5)

# Part of Speech

In [ ]:
#Sample 80k 'academic' messages, reducing n from 100k for spaCy's part of speech tagger
acad_texts = acaDF[acaDF['TEXTENGLISH'].notnull()].sample(80000, random_state=42)
acad_texts = acad_texts['TEXTENGLISH'].str.replace('\n', '', regex=True).str.replace('\s+', ' ', regex=True).tolist()
print(len(acad_texts))

In [ ]:
#Sample 80k random messages, reducing n from 100k for spaCy's part of speech tagger
rand_texts = ranDF[ranDF['TEXTENGLISH'].notnull()].sample(80000, random_state=42)
rand_texts = rand_texts['TEXTENGLISH'].str.replace('\n', '', regex=True).str.replace('\s+', ' ', regex=True).tolist()
print(len(rand_texts))

In [ ]:
print(acad_texts[0])

In [ ]:
print(rand_texts[0])

In [ ]:
#identify target part of speech tags to flag for printing
target_tags = ['WP$', 'RBS', 'PDT', 'JJS', 'RBR', 'POS', 'VBG', 'VBZ', 'JJR', 'PRP$']
tag_count = {}
for tag in target_tags:
    tag_count[tag] = 0

#Store a list of POS tags for the 'academic' sample
count = 0
acad_pos_tags = []
for message in nlp.pipe(acad_texts):
    flag = True
    for token in message:
        acad_pos_tags.append(token.tag_)
        if token.tag_ in target_tags and tag_count[token.tag_] <= 5 and flag:
            print(token.tag_, message)
            tag_count[token.tag_] += 1
            flag = False

In [ ]:
print(len(acad_pos_tags))
print(type(acad_pos_tags[0]), acad_pos_tags[0])

In [ ]:
#Store a list of POS tags for the random sample
rand_pos_tags = []
for message in nlp.pipe(rand_texts):
    for token in message:
        rand_pos_tags.append(token.tag_)

In [ ]:
print(len(rand_pos_tags))
print(type(rand_pos_tags[0]), rand_pos_tags[0])

In [ ]:
#Clip both samples at 250000 token-level POS tags
import random

acad_pos_tags = random.sample(acad_pos_tags, 2000000)
rand_pos_tags = random.sample(rand_pos_tags, 2000000)
print(len(acad_pos_tags), len(rand_pos_tags))

In [ ]:
#Get POS frequency counts for each corpus
from collections import defaultdict

acad_pos_freq = defaultdict(int)
for tag in acad_pos_tags:
    acad_pos_freq[tag] += 1
    
rand_pos_freq = defaultdict(int)
for tag in rand_pos_tags:
    rand_pos_freq[tag] += 1

### Metric: log ratio of POS tag frequency in acad corpus / frequency in rand corpus <br>
for each POS tag: np.log( n_acad / n_rand )

In [ ]:
#Metric: log ratio of POS tag frequency in acad corpus / frequency in rand corpus
pos_log_ratios = []
for k, v in acad_pos_freq.items():
    for ke, va in rand_pos_freq.items():
        if k == ke:
            pos_log_ratios.append(tuple([k, np.log(v / va)]))

In [ ]:
pos_log_ratios.sort(key=lambda x:np.abs(x[1]), reverse=True)

In [ ]:
#https://github.com/explosion/spaCy/blob/master/spacy/glossary.py

GLOSSARY = {
    # POS tags
    # Universal POS Tags
    # http://universaldependencies.org/u/pos/
    "ADJ": "adjective",
    "ADP": "adposition",
    "ADV": "adverb",
    "AUX": "auxiliary",
    "CONJ": "conjunction",
    "CCONJ": "coordinating conjunction",
    "DET": "determiner",
    "INTJ": "interjection",
    "NOUN": "noun",
    "NUM": "numeral",
    "PART": "particle",
    "PRON": "pronoun",
    "PROPN": "proper noun",
    "PUNCT": "punctuation",
    "SCONJ": "subordinating conjunction",
    "SYM": "symbol",
    "VERB": "verb",
    "X": "other",
    "EOL": "end of line",
    "SPACE": "space",
    # POS tags (English)
    # OntoNotes 5 / Penn Treebank
    # https://www.ling.upenn.edu/courses/Fall_2003/ling001/penn_treebank_pos.html
    ".": "punctuation mark, sentence closer",
    ",": "punctuation mark, comma",
    "-LRB-": "left round bracket",
    "-RRB-": "right round bracket",
    "``": "opening quotation mark",
    '""': "closing quotation mark",
    "''": "closing quotation mark",
    ":": "punctuation mark, colon or ellipsis",
    "$": "symbol, currency",
    "#": "symbol, number sign",
    "AFX": "affix",
    "CC": "conjunction, coordinating",
    "CD": "cardinal number",
    "DT": "determiner",
    "EX": "existential there",
    "FW": "foreign word",
    "HYPH": "punctuation mark, hyphen",
    "IN": "conjunction, subordinating or preposition",
    "JJ": "adjective (English), other noun-modifier (Chinese)",
    "JJR": "adjective, comparative",
    "JJS": "adjective, superlative",
    "LS": "list item marker",
    "MD": "verb, modal auxiliary",
    "NIL": "missing tag",
    "NN": "noun, singular or mass",
    "NNP": "noun, proper singular",
    "NNPS": "noun, proper plural",
    "NNS": "noun, plural",
    "PDT": "predeterminer",
    "POS": "possessive ending",
    "PRP": "pronoun, personal",
    "PRP$": "pronoun, possessive",
    "RB": "adverb",
    "RBR": "adverb, comparative",
    "RBS": "adverb, superlative",
    "RP": "adverb, particle",
    "TO": 'infinitival "to"',
    "UH": "interjection",
    "VB": "verb, base form",
    "VBD": "verb, past tense",
    "VBG": "verb, gerund or present participle",
    "VBN": "verb, past participle",
    "VBP": "verb, non-3rd person singular present",
    "VBZ": "verb, 3rd person singular present",
    "WDT": "wh-determiner",
    "WP": "wh-pronoun, personal",
    "WP$": "wh-pronoun, possessive",
    "WRB": "wh-adverb",
    "SP": "space (English), sentence-final particle (Chinese)",
    "ADD": "email",
    "NFP": "superfluous punctuation",
    "GW": "additional word in multi-word expression",
    "XX": "unknown",
    "BES": 'auxiliary "be"',
    "HVS": 'forms of "have"',
    "_SP": "whitespace"
}

In [ ]:
#positive indicates that the given POS is 'x' times more frequent in the academic corpus than the random corpus
#negative indicates that the given POS is 'x' times less frequent in the academic corpus than the random corpus
for tup in pos_log_ratios:
    if tup[1] > 0.2:
        print(GLOSSARY[tup[0]] + ': ' + str(tup[1]))

# Vocabulary

In [ ]:
#Store a list of lemmas for the 'academic' sample
acad_lemmas = []
for message in nlp.pipe(acad_texts):
    for token in message:
        acad_lemmas.append(token.lemma_)
        
print(len(acad_lemmas))
print(type(acad_lemmas[0]), acad_lemmas[0])

In [ ]:
#Store a list of lemmas for the random sample
rand_lemmas = []
for message in nlp.pipe(rand_texts):
    for token in message:
        rand_lemmas.append(token.lemma_)
        
print(len(rand_lemmas))
print(type(rand_lemmas[0]), rand_lemmas[0])

In [ ]:
acad_lemmas = random.sample(acad_lemmas, 2000000)
rand_lemmas = random.sample(rand_lemmas, 2000000)
print(len(acad_lemmas), len(rand_lemmas))

In [ ]:
#Get lemma frequency counts for each corpus
from collections import defaultdict

acad_lemma_freq = defaultdict(int)
for l in acad_lemmas:
    acad_lemma_freq[l] += 1
    
rand_lemma_freq = defaultdict(int)
for l in rand_lemmas:
    rand_lemma_freq[l] += 1

In [ ]:
#Metric: log ratio of lemma frequency in acad corpus / frequency in rand corpus
lemma_log_ratios = []
for l, f in acad_lemma_freq.items():
    for lem, fre in rand_lemma_freq.items():
        if l == lem:
            lemma_log_ratios.append(tuple([l, np.log(f / fre)]))

In [ ]:
lemma_log_ratios.sort(key=lambda x:np.abs(x[1]), reverse=True)

In [ ]:
#positive indicates that the given lemma is 'x' times more frequent in the academic corpus than the random corpus
for tup in lemma_log_ratios:
    if tup[1] > 2: #more than twice as frequent in the 'academic' messages
        print(tup[0] + ': ' + str(tup[1]))